# AI 生成 SPICE 比較實驗 v2：加入自動除錯迴圈
**本機小模型 vs 雲端大模型：qwen2.5-coder:7b / qwen3.5:9b / Gemini 3.5 Flash**

v2 新增：
- **自動除錯迴圈**：netlist 模擬失敗或沒達規格時，把 ngspice 錯誤訊息／量測結果回傳給 AI 請它修正，最多 `MAX_FIX_ROUNDS` 輪
- **重複實驗**：每個組合跑 `REPEAT` 次，才能算通過率
- 結果存到 `我的雲端硬碟/ai_spice_results_v2`（不會蓋掉第一版）

研究問題：**小模型有了錯誤回饋，能不能追上大模型？哪些錯誤修得好、哪些修不好？**

**使用前：**
1. 上方選單「執行階段 → 變更執行階段類型 → T4 GPU」
2. 左側 🔑「密鑰 (Secrets)」新增 `GEMINI_API_KEY`（到 https://aistudio.google.com 申請），並打開「筆記本存取權」。沒填的話會自動跳過 Gemini。
3. 「執行階段 → 全部執行」

## 0. 掛載 Google Drive（先做！）
結果會**每跑完一題就直接存進** `我的雲端硬碟/ai_spice_results_v2`。中途斷線或關機也不會遺失，重新「全部執行」會從斷掉的地方接著跑。
執行後會跳出授權視窗，按允許。**授權完成再離開電腦。**

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

## 1. 安裝 ngspice、Ollama、Gemini SDK

In [ ]:
!apt-get -qq update > /dev/null && apt-get -qq install -y ngspice zstd fonts-noto-cjk > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh
!pip -q install google-genai
!ngspice --version | head -n 2
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. 背景啟動 Ollama 伺服器

In [ ]:
import subprocess, time, requests
ollama_proc = subprocess.Popen(["ollama", "serve"], stdout=open("ollama.log", "w"), stderr=subprocess.STDOUT)
for _ in range(60):
    try:
        requests.get("http://localhost:11434", timeout=2); print("Ollama 已啟動"); break
    except Exception:
        time.sleep(1)
else:
    raise RuntimeError("Ollama 啟動失敗，請看 ollama.log")

## 3. 下載本機模型（約 5～10 分鐘）

In [ ]:
!ollama pull qwen2.5-coder:7b
!ollama pull qwen3.5:9b
!ollama list

## 4. 實驗設定
- `MODELS`：要比較的模型（可自行增刪）
- `REPEAT`：每個組合重複幾次（3 → 3 模型 × 3 題 × 2 Prompt × 3 = 54 組）
- `MAX_FIX_ROUNDS`：失敗後最多讓 AI 修正幾輪（0 = 不修正，等同第一版）
- 預估時間 3–5 小時（qwen 每次 10–200 秒；Gemini 塞車時會等待重試）

In [ ]:
import os
GEMINI_MODEL = "gemini-3.5-flash"   # 3.8 Flash 太常 503/504，第一版最後也是用 3.5
REPEAT = 3
MAX_FIX_ROUNDS = 3
TEMPERATURE = 0.2

MODELS = [
    {"label": "qwen25-coder-7b", "backend": "ollama", "model": "qwen2.5-coder:7b", "think": None},
    {"label": "qwen35-9b",       "backend": "ollama", "model": "qwen3.5:9b",       "think": False},  # 關閉思考模式加速
    {"label": "gemini",           "backend": "gemini", "model": GEMINI_MODEL},
]

try:
    from google.colab import userdata
    GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")
except Exception:
    GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY")
if not GEMINI_API_KEY:
    print("⚠️ 沒有 GEMINI_API_KEY，略過 Gemini")
    MODELS = [m for m in MODELS if m["backend"] != "gemini"]
print("本次比較：", [m["label"] for m in MODELS])

## 5. 題目與 Prompt
三題由易到難；每題兩種 Prompt：
- **simple**：只給規格
- **detailed**：加上拓樸、模型參數、手算步驟與語法規則

兩種 Prompt 都規定介面（輸入節點 `in`、輸出節點 `out`），這樣才能自動驗證。

In [ ]:
RULES = '''
Output requirements (must follow):
- Output exactly ONE ngspice-compatible SPICE netlist inside a ```spice code block.
- The input AC source must drive node `in` with `AC 1`. The output node must be named `out`.
- Define every device model with a .model line inside the netlist (no .include / .lib).
- Do not include a .control block.
'''

TASKS = {
    "rc_lpf": {
        "name": "RC 低通濾波器 fc=1kHz",
        "simple": "Design a first-order passive RC low-pass filter with a -3 dB cutoff frequency of 1 kHz.",
        "detailed": '''You are an analog circuit designer. Design a first-order passive RC low-pass filter.
Spec: -3 dB cutoff frequency fc = 1 kHz, passband gain 0 dB.
Steps: use fc = 1/(2*pi*R*C); pick a standard capacitor (e.g. 100 nF) and compute R; write the hand calculation as SPICE comments (* ...).
Topology: V1 in 0 AC 1 -> series R from in to out -> C from out to ground. No load resistor.''',
        "spec": "fc 在 900–1100 Hz",
        "spec_en": "-3 dB cutoff frequency between 900 Hz and 1100 Hz (target 1 kHz)",
    },
    "cs_amp": {
        "name": "共源極放大器 |Av|=10",
        "simple": "Design a common-source NMOS amplifier with a small-signal voltage gain magnitude of 10 (20 dB) at 1 kHz, VDD = 5 V.",
        "detailed": '''You are an analog circuit designer. Design a single-stage common-source NMOS amplifier.
Spec: |Av| = 10 (20 dB) at 1 kHz, VDD = 5 V, no external load.
Use this model: .model NMOS1 nmos level=1 vto=0.7 kp=200u lambda=0.01
Topology: resistor-divider gate bias, 10uF input coupling capacitor from `in` to the gate, drain resistor RD from VDD to drain, source to ground, node `out` = drain (no output coupling capacitor).
Steps: choose ID and VGS, compute gm = 2*ID/(VGS-VTO), choose RD so that gm*RD ~= 10, check the MOSFET stays in saturation (VDS > VGS-VTO). Write the hand calculation as SPICE comments (* ...).
Input source: V1 in 0 DC 0 AC 1.''',
        "spec": "1 kHz 增益 17–23 dB",
        "spec_en": "small-signal gain at 1 kHz between 17 dB and 23 dB (target 20 dB, |Av| = 10)",
    },
    "opamp": {
        "name": "兩級 CMOS OPA 開迴路增益 ≥ 60 dB",
        "simple": "Design a two-stage CMOS operational amplifier (differential input pair + common-source second stage, Miller compensated) with open-loop DC gain of at least 60 dB, VDD = 5 V. The non-inverting input is node `in`.",
        "detailed": '''You are an analog IC designer. Design a two-stage Miller-compensated CMOS op amp.
Spec: open-loop gain >= 60 dB at low frequency, VDD = 5 V, CL = 1 pF.
Use these models:
.model NMOS1 nmos level=1 vto=0.7 kp=110u lambda=0.04
.model PMOS1 pmos level=1 vto=-0.7 kp=50u lambda=0.05
Topology: NMOS differential pair with PMOS current-mirror load, NMOS tail current source biased by a current mirror (e.g. 20uA reference), PMOS common-source second stage with NMOS current-source load, Miller capacitor Cc (~2pF) from first-stage output to `out`.
Open-loop AC test bench (important, so the output does not saturate):
  Vin in 0 DC 2.5 AC 1   (non-inverting input = node in)
  inverting input node inn: connect inn to out through a 1G H inductor (L 1 out inn 1G) and inn to ground through a 1 F capacitor (C 1 inn 0 1)
Steps: give W/L for every transistor, check all transistors saturated, write the hand calculation as SPICE comments (* ...).''',
        "spec": "10 Hz 增益 ≥ 60 dB",
        "spec_en": "open-loop gain at 10 Hz of at least 60 dB",
    },
}
PROMPT_TYPES = ["simple", "detailed"]

## 6. 工具函式：呼叫模型、擷取 netlist、ngspice 模擬、量測

In [ ]:
import re, time, json, subprocess, pathlib
import numpy as np

OUT = pathlib.Path("/content/drive/MyDrive/ai_spice_results_v2")
if not pathlib.Path("/content/drive/MyDrive").exists():
    print("⚠️ 沒掛載 Drive，結果只存在 /content/results（斷線會遺失）")
    OUT = pathlib.Path("/content/results")
OUT.mkdir(parents=True, exist_ok=True); print("結果存放位置：", OUT)
def W(stem, ext):  # 檔名含 "qwen2.5" 的點，不能用 with_suffix
    return stem.parent / (stem.name + ext)
# 讓 ngspice 盡量相容 LTspice / PSpice 語法
(OUT / ".spiceinit").write_text("set ngbehavior=ltpsa\n")

# ---------- 呼叫模型 ----------
# messages = [{"role": "user"/"assistant", "content": ...}, ...]（多輪對話，用於除錯迴圈）
def gen_ollama(m, messages):
    body = {"model": m["model"], "stream": False, "messages": messages,
            "options": {"temperature": TEMPERATURE, "num_ctx": 16384, "num_predict": 4000}}  # 限制長度，避免模型無限重複
    if m.get("think") is not None:
        body["think"] = m["think"]
    r = requests.post("http://localhost:11434/api/chat", json=body, timeout=600)
    r.raise_for_status()
    return r.json()["message"]["content"]

_gclient = None
def gen_gemini(m, messages):
    global _gclient
    from google import genai
    from google.genai import types
    if _gclient is None:  # 每次請求最多等 3 分鐘，避免卡死
        _gclient = genai.Client(api_key=GEMINI_API_KEY, http_options=types.HttpOptions(timeout=180000))
    contents = [{"role": "model" if x["role"] == "assistant" else "user", "parts": [{"text": x["content"]}]}
                for x in messages]
    for i in range(6):  # 免費額度有速率限制，失敗就等一下重試
        try:
            resp = _gclient.models.generate_content(
                model=m["model"], contents=contents,
                config=types.GenerateContentConfig(temperature=TEMPERATURE))
            return resp.text
        except Exception as e:
            print("   Gemini 錯誤，重試中：", str(e)[:120]); time.sleep(30 * (i + 1))
    raise RuntimeError("Gemini 連續失敗")

def generate(m, messages):
    return gen_ollama(m, messages) if m["backend"] == "ollama" else gen_gemini(m, messages)

# ---------- 從回答擷取 netlist ----------
ELEM = re.compile(r"^[RCLVIMQDEGFHXKJBSW][\w.]*\s+\S+\s+\S+", re.I)
def extract_netlist(text):
    text = re.sub(r"<think>.*?</think>", "", text, flags=re.S)
    blocks = re.findall(r"```[ \t]*[\w+-]*[ \t]*\n(.*?)```", text, flags=re.S)
    if blocks:
        def score(b): return sum(bool(ELEM.match(l.strip())) for l in b.splitlines())
        best = max(blocks, key=score)
        return best if score(best) > 0 else None
    lines = [l for l in text.splitlines() if ELEM.match(l.strip()) or l.strip().startswith((".", "*", "+"))]
    return "\n".join(lines) if sum(bool(ELEM.match(l.strip())) for l in lines) >= 2 else None

# ---------- 清理並加上自動量測 ----------
DROP = (".control", ".endc", ".ac", ".tran", ".op", ".dc", ".noise", ".print", ".plot",
        ".probe", ".backanno", ".meas", ".measure", ".save", ".option", ".options", ".step")
def prepare(netlist, datfile):
    lines, in_ctrl = [], False
    for l in netlist.splitlines():
        s = l.strip().lower()
        if s.startswith(".control"): in_ctrl = True; continue
        if s.startswith(".endc"): in_ctrl = False; continue
        if in_ctrl or s == ".end" or s.startswith(DROP): continue
        lines.append(l.rstrip())
    while lines and not lines[0].strip(): lines.pop(0)
    # ngspice 把第一行當標題：若第一行是元件，就先補一行標題
    first = lines[0].strip() if lines else ""
    is_elem = bool(ELEM.match(first)) and bool(re.search(r"\d", first)) and not re.search(
        r"filter|amplifier|amp\b|design|circuit|netlist|op\s*amp|stage", first, re.I)
    if lines and is_elem:
        lines.insert(0, "* AI generated netlist")
    elif lines and not lines[0].lstrip().startswith("*"):
        lines[0] = "* " + lines[0]
    lines += [".control", "set wr_singlescale", "ac dec 50 1 1e9",
              f"wrdata {datfile} vdb(out)", "quit", ".endc", ".end"]
    return "\n".join(lines) + "\n"

def simulate(netlist, stem):
    dat = f"{stem.name}.dat"
    cir = W(stem, ".cir"); cir.write_text(prepare(netlist, dat))
    (stem.parent / dat).unlink(missing_ok=True)
    try:
        p = subprocess.run(["ngspice", "-b", cir.name], cwd=stem.parent,
                           capture_output=True, text=True, timeout=120)
        log = p.stdout + p.stderr
    except subprocess.TimeoutExpired:
        return None, "simulation timeout"
    W(stem, ".log").write_text(log)
    datp = stem.parent / dat
    if not datp.exists():
        L = [l.strip() for l in log.splitlines() if l.strip()]
        idx = [i for i, l in enumerate(L) if "rror" in l or "not found" in l or "unknown" in l.lower()]
        msg = " / ".join(L[idx[0]:idx[0] + 3]) if idx else (L[-1] if L else "no output")
        return None, msg[:300]
    d = np.atleast_2d(np.loadtxt(datp))
    return (d[:, 0], d[:, -1]), ""  # 最後一欄 = vdb(out)

# ---------- 依題目量測 ----------
def gain_at(f, g, f0): return float(np.interp(np.log10(f0), np.log10(f), g))
def cross(f, g, level):
    idx = np.where((g[:-1] >= level) & (g[1:] < level))[0]
    if len(idx) == 0: return None
    i = idx[0]; t = (g[i] - level) / (g[i] - g[i + 1])
    return float(10 ** (np.log10(f[i]) + t * (np.log10(f[i + 1]) - np.log10(f[i]))))

def measure(task, f, g):
    if task == "rc_lpf":
        fc = cross(f, g, g[0] - 3)
        return (f"fc={fc:.0f}Hz" if fc else "no -3dB point"), bool(fc and 900 <= fc <= 1100)
    if task == "cs_amp":
        a = gain_at(f, g, 1e3)
        return f"gain@1k={a:.2f}dB", 17 <= a <= 23
    if task == "opamp":
        a = gain_at(f, g, 10); u = cross(f, g, 0)
        return f"gain@10Hz={a:.1f}dB, UGBW={u/1e6:.2f}MHz" if u else f"gain@10Hz={a:.1f}dB", a >= 60

def evaluate(tk, reply, stem):
    '''回傳 (status, result, passed, error, curve)'''
    if not reply: return "gen_error", "", False, "", None
    net = extract_netlist(reply)
    if net is None: return "no_netlist", "", False, "", None
    W(stem, ".sp").write_text(net)
    res, err = simulate(net, stem)
    if res is None: return "sim_error", "", False, err, None
    r, ok = measure(tk, *res)
    return ("pass" if ok else "spec_fail"), r, bool(ok), "", res

def feedback(tk, status, result, err):
    '''把失敗原因寫成給 AI 的修正要求（模擬工程師看 log 再改的流程）'''
    if status == "no_netlist":
        msg = "I could not find a SPICE netlist inside a ```spice code block in your answer."
    elif status == "sim_error":
        msg = f"ngspice failed to simulate your netlist. The error message was:\n{err}"
    else:
        msg = (f"Your netlist simulates in ngspice, but the measured result is: {result}.\n"
               f"The spec requires: {TASKS[tk]['spec_en']}.")
    return msg + "\nFind the cause, fix the design, and output the complete corrected netlist.\n" + RULES

def check_file(path, task):
    '''手動修正 .sp 後重新驗證用：check_file("/content/results/xxx.sp", "cs_amp")'''
    path = pathlib.Path(path)
    res, err = simulate(path.read_text(), path.with_name(path.stem))
    if res is None: print("❌ 模擬失敗：", err); return
    print(*measure(task, *res))

## 6.5 除錯：查看某一次的 netlist 與 ngspice 錯誤

In [ ]:
def show(model, task, prompt, rep=1, rnd=0):
    '''rnd = 第幾輪（0 = 第一次生成，1 = 第一次修正 ...）'''
    stem = OUT / f"{model}__{task}__{prompt}__r{rep}__fix{rnd}"
    cir = W(stem, ".cir")
    if cir.exists():
        print("===== 送進 ngspice 的 netlist（含行號）=====")
        for i, l in enumerate(cir.read_text().splitlines(), 1): print(f"{i:3d}| {l}")
    if W(stem, ".log").exists():
        print("\n===== ngspice log =====\n" + W(stem, ".log").read_text()[-2500:])
# show("qwen35-9b", "cs_amp", "simple", rep=1, rnd=0)

## 7. 開始跑實驗（生成 → 模擬 → 失敗就回饋修正，最多 MAX_FIX_ROUNDS 輪）
每一組跑完立刻寫入 Drive 的 `results.jsonl` / `results.csv`；斷線後重新「全部執行」會跳過已完成的組合。
檔名 `…__r2__fix1` = 第 2 次重複、第 1 輪修正。

In [ ]:
import pandas as pd
LOG = OUT / "results.jsonl"
done = {}
if LOG.exists():
    for line in LOG.read_text(encoding="utf-8").splitlines():
        r = json.loads(line); done[(r["model"], r["task"], r["prompt"], r["rep"])] = r

rows, curves = [], {}
total = len(MODELS) * len(TASKS) * len(PROMPT_TYPES) * REPEAT; n = 0
for m in MODELS:
    for tk, t in TASKS.items():
        for pt in PROMPT_TYPES:
            for rep in range(1, REPEAT + 1):
                n += 1; key = (m["label"], tk, pt, rep)
                tag = f"[{n}/{total}] {m['label']} | {tk} | {pt} | r{rep}"
                if key in done and done[key]["final_status"] != "gen_error":   # 已完成 → 跳過
                    rows.append(done[key]); datp = W(OUT / done[key]["final_stem"], ".dat")
                    if datp.exists():
                        d = np.atleast_2d(np.loadtxt(datp)); curves[key] = (d[:, 0], d[:, -1])
                    print(tag, "已完成，略過：", done[key]["history"]); continue

                prompt = t[pt] + "\n" + RULES
                msgs = [{"role": "user", "content": prompt}]
                hist, t_total = [], 0.0
                for rnd in range(MAX_FIX_ROUNDS + 1):
                    stem = OUT / f"{m['label']}__{tk}__{pt}__r{rep}__fix{rnd}"
                    t0 = time.time()
                    try:
                        reply = generate(m, msgs)
                    except Exception as e:
                        reply = ""; print("   生成失敗：", str(e)[:120])
                    t_total += time.time() - t0
                    W(stem, ".txt").write_text("\n\n".join(f"===== {x['role'].upper()} =====\n{x['content']}" for x in msgs)
                                               + f"\n\n===== ASSISTANT =====\n{reply}")
                    st, r, ok, err, res = evaluate(tk, reply, stem)
                    hist.append(st)
                    if res is not None: curves[key] = res
                    if ok or st == "gen_error" or rnd == MAX_FIX_ROUNDS: break
                    # 失敗 → 保留原題目 + 這次的回答 + 錯誤回饋，請 AI 修正
                    msgs = [msgs[0], {"role": "assistant", "content": reply},
                            {"role": "user", "content": feedback(tk, st, r, err)}]

                row = dict(model=m["label"], task=tk, prompt=pt, rep=rep,
                           first_status=hist[0], final_status=st,
                           passed_first=(hist[0] == "pass"), passed_final=ok,
                           pass_round=(rnd if ok else None), rounds_used=rnd,
                           history=" → ".join(hist), final_result=r, final_error=err,
                           gen_time_s=round(t_total, 1), final_stem=stem.name, manual_fixes="")
                print(tag, "|", row["history"], "|", r or err[:70], f"({t_total:.0f}s)")
                rows.append(row)
                with open(LOG, "a", encoding="utf-8") as fp: fp.write(json.dumps(row, ensure_ascii=False) + "\n")
                pd.DataFrame(rows).to_csv(OUT / "results.csv", index=False, encoding="utf-8-sig")

df = pd.DataFrame(rows)
df.to_csv(OUT / "results.csv", index=False, encoding="utf-8-sig")
df

## 8. 統計結果
- **passed_first**：第一次生成就通過（等同第一版實驗）
- **passed_final**：經過最多 `MAX_FIX_ROUNDS` 輪修正後通過
- **累積通過率**：第 k 輪（含）以前通過的比例 → 看回饋迴圈有沒有用
- 狀態：`pass` 通過／`spec_fail` 跑得動但沒達規格／`sim_error` ngspice 報錯／`no_netlist` 沒產出 netlist／`gen_error` API 失敗

In [ ]:
R = MAX_FIX_ROUNDS
df["pass_round_n"] = pd.to_numeric(df["pass_round"], errors="coerce")
recs = []
for (mdl, pt), g in df.groupby(["model", "prompt"]):
    for k in range(R + 1):
        recs.append(dict(model=mdl, prompt=pt, round=k, pass_rate=100 * (g["pass_round_n"] <= k).mean()))
cum = pd.DataFrame(recs)
cum.to_csv(OUT / "cumulative_pass_rate.csv", index=False, encoding="utf-8-sig")

print("=== 1. 各輪累積通過率 (%)：欄 = 第幾輪修正（0 = 第一次生成）===")
display(cum.pivot_table(index=["model", "prompt"], columns="round", values="pass_rate").round(0))
print("=== 2. 第一次 vs 修正後 通過率 (%) ===")
display(df.groupby("model")[["passed_first", "passed_final"]].mean().mul(100).round(0))
print("=== 3. 各題 × 模型：第一次 → 修正後 (%) ===")
display(df.pivot_table(index="task", columns="model", values=["passed_first", "passed_final"], aggfunc="mean").mul(100).round(0))
print("=== 4. 第一次生成的失敗類型（次數）===")
display(pd.crosstab(df["model"], df["first_status"]))
print("=== 5. 修正後的最終狀態（次數）===")
display(pd.crosstab(df["model"], df["final_status"]))
print("=== 6. 最後有通過的，平均在第幾輪通過 ===")
display(df[df["passed_final"] == True].groupby("model")["pass_round_n"].mean().round(2))
print("=== 7. 修好了哪些錯誤：第一次失敗類型 → 最終狀態（次數）===")
fixed = df[df["first_status"] != "pass"]
display(pd.crosstab([fixed["model"], fixed["first_status"]], fixed["final_status"]))

## 9. 圖表（中文字型，可直接放簡報）

In [ ]:
import glob, matplotlib.pyplot as plt, matplotlib.font_manager as fm
for p in glob.glob("/usr/share/fonts/opentype/noto/NotoSansCJK*.ttc"): fm.fontManager.addfont(p)
plt.rcParams["font.family"] = ["Noto Sans CJK JP", "DejaVu Sans"]
COLOR = {"qwen25-coder-7b": "#8c8c8c", "qwen35-9b": "#e8710a", "gemini": "#1a73e8"}
NAME = {"qwen25-coder-7b": "qwen2.5-coder 7B", "qwen35-9b": "qwen3.5 9B", "gemini": "Gemini 3.5 Flash"}

# (a) 回饋迴圈效果：累積通過率 vs 修正輪數
fig, ax = plt.subplots(figsize=(8, 4.8))
for (mdl, pt), g in cum.groupby(["model", "prompt"]):
    ax.plot(g["round"], g["pass_rate"], marker="o", lw=2, color=COLOR.get(mdl),
            ls="-" if pt == "detailed" else "--", label=f"{NAME.get(mdl, mdl)} / {pt}")
ax.set_xticks(range(R + 1)); ax.set_ylim(-5, 105)
ax.set_xlabel("修正輪數（0 = 第一次生成）"); ax.set_ylabel("累積通過率 (%)")
ax.set_title("自動除錯迴圈：每多修一輪，通過率提升多少？"); ax.grid(alpha=0.3); ax.legend(fontsize=8)
fig.tight_layout(); fig.savefig(OUT / "plot_fix_rounds.png", dpi=200); plt.show()

# (b) 各題：第一次 vs 修正後
fig, axes = plt.subplots(1, len(TASKS), figsize=(13, 4), sharey=True)
for ax, tk in zip(axes, TASKS):
    g = df[df["task"] == tk].groupby("model")[["passed_first", "passed_final"]].mean().mul(100)
    g = g.reindex([m["label"] for m in MODELS])
    x = np.arange(len(g))
    ax.bar(x - 0.2, g["passed_first"], 0.4, label="第一次", color="#c6dafc")
    ax.bar(x + 0.2, g["passed_final"], 0.4, label=f"修正 ≤{R} 輪後", color="#1a73e8")
    ax.set_xticks(x); ax.set_xticklabels([NAME.get(i, i) for i in g.index], fontsize=8)
    ax.set_title(TASKS[tk]["name"]); ax.set_ylim(0, 105); ax.grid(axis="y", alpha=0.3)
axes[0].set_ylabel("通過率 (%)"); axes[0].legend(fontsize=8)
fig.tight_layout(); fig.savefig(OUT / "plot_first_vs_final.png", dpi=200); plt.show()

# (c) 頻率響應（每個組合取第 1 次重複的最終版本）
for tk, t in TASKS.items():
    fig, ax = plt.subplots(figsize=(9, 5))
    for (ml, k, pt, rp), (f, gdb) in curves.items():
        if k == tk and rp == 1:
            ax.semilogx(f, gdb, color=COLOR.get(ml), ls="-" if pt == "detailed" else "--", lw=2,
                        label=f"{NAME.get(ml, ml)} / {pt}")
    ax.set_title(f"{t['name']}　規格：{t['spec']}"); ax.set_xlabel("頻率 (Hz)"); ax.set_ylabel("增益 (dB)")
    ax.grid(True, which="both", alpha=0.3)
    if ax.lines: ax.legend(fontsize=8)
    fig.tight_layout(); fig.savefig(OUT / f"plot_{tk}.png", dpi=200); plt.show()

## 10. 手動修正後重新驗證（選用）
打開 Drive `ai_spice_results_v2/` 裡失敗的 `.sp`，修正後執行下面這格，並把修改處數量填進 `results.csv` 的 `manual_fixes` 欄。

In [ ]:
# check_file("/content/drive/MyDrive/ai_spice_results_v2/qwen25-coder-7b__cs_amp__simple__r1.sp", "cs_amp")

## 11. 打包並自動關閉執行階段（省運算單元）

In [ ]:
import shutil
shutil.make_archive(str(OUT.parent / "ai_spice_results_v2"), "zip", OUT)
print("已打包：", OUT.parent / "ai_spice_results_v2.zip")
AUTO_SHUTDOWN = True   # 跑完自動釋放 GPU；想留著執行階段就改 False
if AUTO_SHUTDOWN:
    from google.colab import runtime
    runtime.unassign()